In [ ]:
# importing all necessary libraries
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

from tqdm.notebook import tqdm as tqdm

import plotly.graph_objs as go #visualization library
import random
import os
import time
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import gc
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
import multiprocessing as mp
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline
from datetime import datetime
from torch.utils.data import Dataset, DataLoader

In [ ]:
from pandas.tseries import offsets
from pandas.tseries.frequencies import to_offset
from typing import List

In [ ]:
class TimeFeature:
    def __init__(self):
        pass

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        pass

    def __repr__(self):
        return self.__class__.__name__ + "()"


class SecondOfMinute(TimeFeature):
    """Minute of hour encoded as value between [-0.5, 0.5]"""

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        return index.second / 59.0 - 0.5


class MinuteOfHour(TimeFeature):
    """Minute of hour encoded as value between [-0.5, 0.5]"""

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        return index.minute / 59.0 - 0.5


class HourOfDay(TimeFeature):
    """Hour of day encoded as value between [-0.5, 0.5]"""

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        return index.hour / 23.0 - 0.5


class DayOfWeek(TimeFeature):
    """Hour of day encoded as value between [-0.5, 0.5]"""

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        return index.dayofweek / 6.0 - 0.5


class DayOfMonth(TimeFeature):
    """Day of month encoded as value between [-0.5, 0.5]"""

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        return (index.day - 1) / 30.0 - 0.5


class DayOfYear(TimeFeature):
    """Day of year encoded as value between [-0.5, 0.5]"""

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        return (index.dayofyear - 1) / 365.0 - 0.5


class MonthOfYear(TimeFeature):
    """Month of year encoded as value between [-0.5, 0.5]"""

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        return (index.month - 1) / 11.0 - 0.5


class WeekOfYear(TimeFeature):
    """Week of year encoded as value between [-0.5, 0.5]"""

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        return (index.isocalendar().week - 1) / 52.0 - 0.5


def time_features_from_frequency_str(freq_str: str) -> List[TimeFeature]:
    """
    Returns a list of time features that will be appropriate for the given frequency string.
    Parameters
    ----------
    freq_str
        Frequency string of the form [multiple][granularity] such as "12H", "5min", "1D" etc.
    """

    features_by_offsets = {
        offsets.YearEnd: [],
        offsets.QuarterEnd: [MonthOfYear],
        offsets.MonthEnd: [MonthOfYear],
        offsets.Week: [DayOfMonth, WeekOfYear],
        offsets.Day: [DayOfWeek, DayOfMonth, DayOfYear],
        offsets.BusinessDay: [DayOfWeek, DayOfMonth, DayOfYear],
        offsets.Hour: [HourOfDay, DayOfWeek, DayOfMonth, DayOfYear],
        offsets.Minute: [
            MinuteOfHour,
            HourOfDay,
            DayOfWeek,
            DayOfMonth,
            DayOfYear,
        ],
        offsets.Second: [
            SecondOfMinute,
            MinuteOfHour,
            HourOfDay,
            DayOfWeek,
            DayOfMonth,
            DayOfYear,
        ],
    }

    offset = to_offset(freq_str)

    for offset_type, feature_classes in features_by_offsets.items():
        if isinstance(offset, offset_type):
            return [cls() for cls in feature_classes]

    supported_freq_msg = f"""
    Unsupported frequency {freq_str}
    The following frequencies are supported:
        Y   - yearly
            alias: A
        M   - monthly
        W   - weekly
        D   - daily
        B   - business days
        H   - hourly
        T   - minutely
            alias: min
        S   - secondly
    """
    raise RuntimeError(supported_freq_msg)


def time_features(dates, freq='h'):
    return np.vstack([feat(dates) for feat in time_features_from_frequency_str(freq)])

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

In [ ]:
class Dataset_ETT_hour(Dataset):
    def __init__(self, root_path, flag='train', size=None,
                 features='S', data_path='ETTh1.csv',
                 target='OT', scale=True, timeenc=0, freq='h', seasonal_patterns=None):
        # size [seq_len, label_len, pred_len]
        # info
        if size == None:
            self.seq_len = 24 * 4 * 4
            self.label_len = 24 * 4
            self.pred_len = 24 * 4
        else:
            self.seq_len = size[0]
            self.label_len = size[1]
            self.pred_len = size[2]
        # init
        assert flag in ['train', 'test', 'val']
        type_map = {'train': 0, 'val': 1, 'test': 2}
        self.set_type = type_map[flag]

        # M:multivariate predict multivariate, S:univariate predict univariate, MS:multivariate predict univariate'
        self.features = features
        self.target = target
        self.scale = scale
        self.timeenc = timeenc
        self.freq = freq

        self.root_path = root_path
        self.data_path = data_path
        self.__read_data__()

    def __read_data__(self):
        self.scaler = StandardScaler()
        df_raw = pd.read_csv(os.path.join(self.root_path,
                                          self.data_path))
        border1s = [0, 12 * 30 * 24 - self.seq_len, 12 * 30 * 24 + 4 * 30 * 24 - self.seq_len]
        border2s = [12 * 30 * 24, 12 * 30 * 24 + 4 * 30 * 24, 12 * 30 * 24 + 8 * 30 * 24]
        border1 = border1s[self.set_type]
        border2 = border2s[self.set_type]
        if self.features == 'M' or self.features == 'MS':
            cols_data = df_raw.columns[1:]
            df_data = df_raw[cols_data]
        elif self.features == 'S':
            df_data = df_raw[[self.target]]
        if self.scale:
            train_data = df_data[border1s[0]:border2s[0]]
            self.scaler.fit(train_data.values)
            data = self.scaler.transform(df_data.values)
        else:
            data = df_data.values
        df_stamp = df_raw[['date']][border1:border2]
        df_stamp['date'] = pd.to_datetime(df_stamp.date)
        if self.timeenc == 0:
            df_stamp['month'] = df_stamp.date.apply(lambda row: row.month, 1)
            df_stamp['day'] = df_stamp.date.apply(lambda row: row.day, 1)
            df_stamp['weekday'] = df_stamp.date.apply(lambda row: row.weekday(), 1)
            df_stamp['hour'] = df_stamp.date.apply(lambda row: row.hour, 1)
            data_stamp = df_stamp.drop(['date'], 1).values
        elif self.timeenc == 1:
            data_stamp = time_features(pd.to_datetime(df_stamp['date'].values), freq=self.freq)
            data_stamp = data_stamp.transpose(1, 0)

        self.data_x = data[border1:border2]
        self.data_y = data[border1:border2]
        self.data_stamp = data_stamp

    def __getitem__(self, index):
        s_begin = index
        s_end = s_begin + self.seq_len
        r_begin = s_end - self.label_len
        r_end = r_begin + self.label_len + self.pred_len

        seq_x = self.data_x[s_begin:s_end]
        seq_y = self.data_y[r_begin:r_end]
        seq_x_mark = self.data_stamp[s_begin:s_end]
        seq_y_mark = self.data_stamp[r_begin:r_end]

        return seq_x, seq_y, seq_x_mark, seq_y_mark

    def __len__(self):
        return len(self.data_x) - self.seq_len - self.pred_len + 1

    def inverse_transform(self, data):
        return self.scaler.inverse_transform(data)

In [ ]:
root_path='ETT-small'
data_path='ETTh2.csv'
df_raw = pd.read_csv(os.path.join(root_path,data_path))

In [ ]:
df_raw.shape

In [ ]:
df_raw.head()

In [ ]:
df_raw['date'] = pd.to_datetime(df_raw.date)

In [ ]:
df_raw['month'] = df_raw.date.apply(lambda row: row.month, 1)
df_raw['day'] = df_raw.date.apply(lambda row: row.day, 1)
df_raw['weekday'] = df_raw.date.apply(lambda row: row.weekday(), 1)
df_raw['hour'] = df_raw.date.apply(lambda row: row.hour, 1)
#data_stamp = df_stamp.drop(['date'], 1).values

In [ ]:
df_raw.head()

In [ ]:
cols_data = df_raw.columns[1:]
df_data = df_raw[cols_data]

In [ ]:
cols_data

In [ ]:
TARGET ="OT"
keep_cols = ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'month', 'day','weekday', 'hour', TARGET]

In [1]:
cols = ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'month', 'day','weekday', 'hour']

In [2]:
len(cols)

10

In [ ]:
df_data.head()

In [ ]:
df_data = df_data[keep_cols]
df_data.shape

In [ ]:
X = df_data.copy()
X

In [ ]:
X.head()

In [ ]:
timesteps = 48
prediction_horizon = 720

In [ ]:
X_data_imputed = np.zeros((len(X), timesteps, X.shape[1]-1))
X_data_no_imputed = np.zeros((len(X), timesteps, X.shape[1]-1))
y_data_imputed = np.zeros((len(X), prediction_horizon, 1))
y_data_no_imputed = np.zeros((len(X), prediction_horizon, 1))

In [ ]:
X_data_imputed.shape ,X_data_no_imputed.shape ,y_data_imputed.shape ,y_data_no_imputed.shape 

In [ ]:
for i, name in enumerate(cols):
    for j in range(timesteps):
        X_data_imputed[:, j, i] = X[name].shift(timesteps - j - 1).fillna(method="bfill")
        X_data_no_imputed[:, j, i] = X[name].shift(timesteps - j - 1)
for j in range(prediction_horizon):
    y_data_imputed[:, j, 0] = X[TARGET].shift(prediction_horizon - j - 1).fillna(method="bfill")
    y_data_no_imputed[:, j, 0] = X[TARGET].shift(prediction_horizon - j - 1)

In [ ]:
X_data_imputed.shape ,X_data_no_imputed.shape ,y_data_imputed.shape ,y_data_no_imputed.shape 

In [ ]:
X_data_no_imputed

In [ ]:
X_data_imputed = X_data_imputed[timesteps:-prediction_horizon]
X_data_no_imputed = X_data_no_imputed[timesteps:-prediction_horizon]
y_data_imputed = y_data_imputed[timesteps:-prediction_horizon]
y_data_no_imputed = y_data_no_imputed[timesteps:-prediction_horizon]

In [ ]:
X_data_imputed.shape ,X_data_no_imputed.shape ,y_data_imputed.shape,y_data_no_imputed.shape

In [ ]:
y_data_no_imputed[np.isnan(y_data_no_imputed)],  y_data_no_imputed[np.isinf(y_data_no_imputed)]

In [ ]:
X_data_no_imputed[np.isnan(X_data_no_imputed)],  X_data_no_imputed[np.isinf(X_data_no_imputed)]

In [ ]:
X_data_no_imputed

# Extract the last observation of each feature

In [ ]:
nb_pats, seq, n_features = X_data_no_imputed.shape
timeseries_last_obs_data = []
for i in range(nb_pats):
    Index_Last=(~np.isnan(X_data_no_imputed[i,:,:])).cumsum(0).argmax(0)
    Last_Indices = np.reshape(Index_Last,(1,n_features))
    Last_Values = np.take_along_axis(X_data_no_imputed[i,:,:], Last_Indices, axis = 0)
    timeseries_last_obs_data.append(np.repeat(Last_Values, seq, axis=0))
last_obs_data=np.stack(timeseries_last_obs_data)

In [ ]:
last_obs_data[np.isnan(last_obs_data)],  last_obs_data[np.isinf(last_obs_data)]

In [ ]:
col_median = np.nanmedian(last_obs_data, axis=0)
last_obs_data_imputed = np.where(np.isnan(last_obs_data), col_median, last_obs_data)

In [ ]:
last_obs_data[np.isnan(last_obs_data)],  last_obs_data[np.isinf(last_obs_data)]

# The frequency observation of each feature -----> np.isfinite

In [ ]:
# isnan
freq_list= []
nb_pats, seq, n_features = X_data_no_imputed.shape
for i in range(nb_pats):
    data_samples=  np.expand_dims(X_data_no_imputed[i,:,:], axis=0)
    nan_counts = np.sum(np.isnan(data_samples), axis=(0, 1))
    freq_list.append(np.repeat(np.expand_dims(nan_counts, axis=0), seq, axis=0))
freqs = np.stack(freq_list)

In [ ]:
freqs, freqs.shape

In [ ]:
import math
import re
import itertools  

import datetime
from collections import namedtuple, defaultdict
from tqdm import tqdm
from itertools import groupby
from operator import itemgetter
from functools import partial
from itertools import combinations

In [ ]:
def forward_imputation(data):
    store_index = []
    for index in  range(len(data)):
        try: 
            if len(data)>1:
                store_index.append([data[index], data[index+1]])
            else:
                store_index.append(data[index])
        except:
            pass
    return store_index
def _forward_with_last_measured(data, hours_data=48):
    cols_data = data.columns.to_list()
    times_hours_data = np.arange(0,hours_data,1)
    notna_cols_indexes, nan_cols_indexes  = defaultdict(list),defaultdict(list)
    indexes_last_indicator = defaultdict(list)
    for col in cols_data:
        notna_cols_indexes[col].append(list(data.loc[pd.notna(data[col]), :].index))
        nan_cols_indexes[col].append(list(data.loc[pd.isna(data[col]), :].index))
        indexes_last_indicator[col].append([data[col].notna()[::-1].idxmax(),
                                      data[col].isna()[::-1].idxmax()])
    notna_cols_indexes_ = {key: list(itertools.chain.from_iterable(value)) 
                                 for key , value in notna_cols_indexes.items() if list(itertools.chain.from_iterable(value))}
    notna_cols_indexes_ ={ key:value  for key , value in notna_cols_indexes_.items() if len(value)!=len(times_hours_data)}

    nan_cols_indexes = {key: list(itertools.chain.from_iterable(value)) 
                                for key , value in nan_cols_indexes.items()}
    nan_cols_indexes_ ={key:value  for key , value in nan_cols_indexes.items() 
                                if len(value)!=len(times_hours_data)}
    nan_cols_indexes_ ={key:value  for key , value in nan_cols_indexes_.items() if value}
    
    indexes_last_indicator = {key: list(itertools.chain.from_iterable(value)) 
                                    for key , value in indexes_last_indicator.items()}
    matrix_indexes_notna = {key:forward_imputation(value) for key, value in notna_cols_indexes_.items()}
    matrix_notna_with_last = {key:list(itertools.chain.from_iterable([matrix_indexes_notna[key], [indexes_last_indicator[key]]])) 
                                 for key in indexes_last_indicator if key in matrix_indexes_notna }
    final_matrix_indexes = defaultdict(list)
    for key, vals in matrix_notna_with_last.items():
        for val in vals:
            if isinstance(val, list):
                final_matrix_indexes[key].append(val)
    matrix_range_indexes_cols = {key:[final_matrix_indexes[key], nan_cols_indexes_[key]] 
                                 for key in nan_cols_indexes_ if key in final_matrix_indexes}
    
    range_indexes_cols_imputed = {}
    for key, value in matrix_range_indexes_cols.items():
        range_values = [[notna_ind,nan] for nan in value[1] 
                      for notna_ind in value[0] if notna_ind[0]<=nan<=notna_ind[1]]
        range_indexes_cols_imputed[key]=range_values
    return range_indexes_cols_imputed

def forward_with_last_measured_value_with_time_elasped_interval(data_copy, mask_data, hrs_used=48):
    resultats = _forward_with_last_measured(data_copy, hours_data=hrs_used)
    data =data_copy.copy()
    mask_forward =mask_data.copy()
    for key , values in resultats.items():
        for k, group in groupby(values, lambda x:x[0]):
            vals_ = list(itertools.chain.from_iterable(list(group)))
            vals_ = np.array([val for val in vals_ if not isinstance(val, list)])
            if k[1]<hrs_used-1:  
                for index in np.arange(k[0]+1, k[1]):
                    data.at[index, key]= data._get_value(k[0], key)
                    mask_forward.at[index, key]= index-k[0]
            else:
                for index in np.arange(k[0]+1, k[1]+1):
                    if pd.isnull(data._get_value(index, key)):
                        #print(index, key, data[key])
                        data.at[index, key]= data._get_value(k[0], key)
                        mask_forward.at[index, key]= index-k[0]
                    else:
                        pass
    return data, mask_forward  

In [ ]:
x_features = cols
x_features

In [ ]:
timesteps

In [ ]:
timedelta, features_inputs= [],[]
for x_data,freq in tqdm(zip(X_data_no_imputed, freqs), total=len(y_data_imputed),
                            desc='Iterating over samples'):
    masked_timeseries = np.where((pd.isnull(x_data)), np.nan, 0)
    features_data = pd.DataFrame(x_data, columns=x_features)
    masked_episode_timeseries = pd.DataFrame(masked_timeseries, columns=x_features)
    data_for_imp,time_elasped_timeseries=forward_with_last_measured_value_with_time_elasped_interval(features_data,
                                                                                                     masked_episode_timeseries,
                                                                                                     hrs_used=timesteps)
    timedelta.append(time_elasped_timeseries.values)
    features_inputs.append(masked_episode_timeseries.values)

In [ ]:
delta_time_data=np.stack(timedelta)
mask_data=np.stack(features_inputs)

In [ ]:
delta_time_data

In [ ]:
delta_time_data[np.isnan(delta_time_data)],  delta_time_data[np.isinf(delta_time_data)]

In [ ]:
data_path, data_path.split(".")[0]

In [ ]:
dataset_name =f"{data_path.split('.')[0]}_{timesteps}_timesteps_AHEAD_p_horizon_{prediction_horizon}".upper()
dataset_name

In [ ]:
dn=f"/home/cissoko-m-1/Documents/{dataset_name}"
if not os.path.exists(dn):
    os.makedirs(dn)
dn, dataset_name

In [ ]:
np.savez(os.path.join(dn,f"data_{timesteps}.npz"),
         last_obs=last_obs_data, 
         last_obs_imputed=last_obs_data_imputed, 
         targets_data=y_data_imputed,
         frequencies_data=freqs, 
         timeseries_data=X_data_no_imputed,
         timeseries_data_imputed=X_data_imputed,
         timedelta_data=delta_time_data) 

In [ ]:
delta_time_data[np.isnan(delta_time_data)] = 999
delta_time_data[np.isinf(delta_time_data)] = 999

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset,TensorDataset
from sklearn.model_selection import KFold, train_test_split

def dataloaders_double_cv_cohort(train_x, train_t,  train_last,  train_freq, train_y, 
                          test_x, test_t,  test_last,  test_freq,  test_y, BATCH_SIZE=128):
        
    def collate_fn(batch):
        return tuple(zip(*batch))

    train_dataset = TensorDataset(torch.tensor(train_x,dtype=torch.float32),
                                  torch.tensor(train_t,dtype=torch.float32),
                                  torch.tensor(train_last,dtype=torch.float32),
                                  torch.tensor(train_freq,dtype=torch.float32),
                                  torch.tensor(train_y,dtype=torch.float32))  
    
    
    valid_dataset = TensorDataset(torch.tensor(test_x,dtype=torch.float32),
                                  torch.tensor(test_t,dtype=torch.float32),
                                  torch.tensor(test_last,dtype=torch.float32),
                                  torch.tensor(test_freq,dtype=torch.float32),
                                  torch.tensor(test_y,dtype=torch.float32))  
    
    
    
    train_data_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
    )

    valid_data_loader = DataLoader(
        valid_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4,
    )
    
    return train_data_loader, valid_data_loader
def cv_fold_splits_cohorts(data,  time_data, last_data,
                           features_freqs, target, n_fold=10):
    from sklearn.model_selection import StratifiedKFold, train_test_split

    test_data_loader, training_data, validation_data = [], [], []

    #  StratifiedKFold preserves class balance across folds
    kfold = KFold(n_splits=n_fold, shuffle=True, random_state=n_fold)
    strat_target = target.squeeze() if target.ndim > 1 else target

    for index, (train_index, test_index) in enumerate(
            kfold.split(data, strat_target)):
        print(f'<------- OUTER FOLD {index + 1} ------->')

        # ── Raw outer split (un-normalized) ──────────────────────────
        x_train,    x_test      = data[train_index],         data[test_index]
        x_train_last, x_test_last = last_data[train_index],  last_data[test_index]
        x_train_freq, x_test_freq = features_freqs[train_index], features_freqs[test_index]
        x_train_t,  x_test_t   = time_data[train_index],    time_data[test_index]
        y_train,    y_test      = target[train_index],       target[test_index]

        # ── Inner split BEFORE normalization and loader creation ──────
        indices = np.arange(len(x_train))
        tr_idx, val_idx = train_test_split(
            indices,
            test_size=0.2,
            random_state=n_fold,
            #stratify=strat_target[train_index]
        )

        x_tr_in,    x_val_in    = x_train[tr_idx],      x_train[val_idx]
        t_tr_in,    t_val_in    = x_train_t[tr_idx],    x_train_t[val_idx]
        last_tr_in, last_val_in = x_train_last[tr_idx], x_train_last[val_idx]
        freq_tr_in, freq_val_in = x_train_freq[tr_idx], x_train_freq[val_idx]
        y_tr_in,    y_val_in    = y_train[tr_idx],       y_train[val_idx]

        # ──  Fit normalization ONLY on the 80% inner training split ─
        def normalize(x, mn, mx):
            scale = np.where(mx - mn == 0, 1.0, mx - mn)
            return (x - mn) / scale

        ts_min  = np.nanmin(x_tr_in,    axis=0)
        ts_max  = np.nanmax(x_tr_in,    axis=0)
        la_min  = np.nanmin(last_tr_in, axis=0)
        la_max  = np.nanmax(last_tr_in, axis=0)
        y_min   = y_tr_in.min();  y_max = y_tr_in.max()

        x_tr_in    = normalize(x_tr_in,    ts_min, ts_max)
        x_val_in   = normalize(x_val_in,   ts_min, ts_max)   #  train stats
        x_te_norm  = normalize(x_test,     ts_min, ts_max)   #  train stats

        last_tr_in  = normalize(last_tr_in,  la_min, la_max)
        last_val_in = normalize(last_val_in, la_min, la_max)
        x_test_last = normalize(x_test_last, la_min, la_max)

        y_tr_in  = normalize(y_tr_in,  y_min, y_max)
        y_val_in = normalize(y_val_in, y_min, y_max)
        y_test   = normalize(y_test,   y_min, y_max)

        # ── Build loaders ─────────────────────────────────────────────
        train_loader = dataloaders_double_cv_cohort(
            x_tr_in, t_tr_in, last_tr_in, freq_tr_in, y_tr_in,
            x_tr_in, t_tr_in, last_tr_in, freq_tr_in, y_tr_in
        )[0]

        val_loader = dataloaders_double_cv_cohort(
            x_tr_in,  t_tr_in,  last_tr_in,  freq_tr_in,  y_tr_in,
            x_val_in,  t_val_in, last_val_in,    #  update x_val_in
            freq_val_in, y_val_in                          #    with true non-imputed
        )[1]                                               #    when available

        #  Test loader built AFTER split & normalization
        _, test_loader = dataloaders_double_cv_cohort(
            x_tr_in,   t_tr_in,   last_tr_in,  freq_tr_in,  y_tr_in,
            x_te_norm, x_test_t, x_test_last, x_test_freq, y_test
        )

        training_data.append(train_loader)
        validation_data.append(val_loader)
        test_data_loader.append(test_loader)

    return test_data_loader, training_data, validation_data, x_tr_in.shape, ts_min, ts_max, y_min , y_max

In [ ]:
TARGETS =np.squeeze(y_data_imputed,axis=-1)
TARGETS

In [ ]:
TARGETS.shape, X_data_no_imputed.shape

In [ ]:
test_loader, train_loader, valid_loader,data_shape, x_min, x_max, y_min, y_max = cv_fold_splits_cohorts(X_data_no_imputed,   
                                                                                                        delta_time_data,last_obs_data, 
                                                                                                        freqs,TARGETS, n_fold=10)

seq_length = X_data_no_imputed.shape[1]
input_dim = X_data_no_imputed.shape[-1]
hidden_dim, output_dim  = 64, TARGETS.shape[-1]
seq_length, input_dim, hidden_dim, output_dim, data_shape
seq_length, input_dim, hidden_dim, output_dim, data_shape
taskname=dataset_name
task=f"{os.path.join(dn, f'{taskname}')}"
if not os.path.exists(task):
    os.makedirs(task)
train_val_loader = random.sample(list(zip(train_loader, valid_loader)), len(train_loader))
np.savez(os.path.join(task, f"train_test_data.npz"), 
            folds_data_test= test_loader,
            folds_data_train_valid= train_val_loader,)
        
np.savez(os.path.join(task, f"data_max_min.npz"), 
         data_max=x_max, data_min=x_min,
         data_targets_max=y_max, data_targets_min=y_min,
         shape_data=data_shape, seq_length=seq_length,
         input_dim=input_dim, output_dim=TARGETS.shape[-1])
input_dim, seq_length, task